In [20]:
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score
import clickhouse_connect
import pandas as pd
import os
from dotenv import load_dotenv

load_dotenv()

True

In [21]:
client = clickhouse_connect.get_client(
    host='localhost',
    port=8143,
    username='default',
    password=os.getenv("CLICKHOUSE_PASSWORD")
)

In [22]:
latitude_and_longitude_encoding_ids = [
    '46ca04f3300b447ab225f808c24c1f2a',
    '60ec2e136e4b7aa88a44078462ab7bf0',
    '69f0f0f1012c1bce555f7de22f849232',
    '6e26eb15061b34b2b95908f46f50c496',
    'dc07b90435445c5e64d7c73cbf766827',
    'de87d594f95f5454e6861e2026e664d0',
]

## Model with Point-in-Time Correctness i.e. lagging features, no leakage

In [34]:
air_pollution_features_df = client.query_df("SELECT * FROM weather_features.feature_air_pollution order by time_of_measurement_utc")
air_pollution_features_df = air_pollution_features_df.dropna()

# Appy one-hot encoding per city and season in the same order every time
air_pollution_features_df['season'] = pd.Categorical(air_pollution_features_df['season'], categories=['Autumn', 'Dry Season', 'Rainy Season', 'Spring', 'Summer', 'Winter'], ordered=True)
air_pollution_features_df['lat_and_long_encoding_id'] = pd.Categorical(air_pollution_features_df['lat_and_long_encoding_id'], categories=latitude_and_longitude_encoding_ids, ordered=True)
air_pollution_features_df = pd.get_dummies(air_pollution_features_df, columns=['lat_and_long_encoding_id'], dtype=int)
air_pollution_features_df = pd.get_dummies(air_pollution_features_df, columns=['season'], dtype=int)

# remove air_pollution_features_row_id. remove time of measurement. 
air_pollution_features_df=air_pollution_features_df.drop(columns="air_pollution_features_row_id")
air_pollution_features_df=air_pollution_features_df.drop(columns="time_of_measurement_utc")
air_pollution_features_df=air_pollution_features_df.drop(columns="time_of_measurement") # relevant time feature

Y=air_pollution_features_df["aqi_6_hour_ahead"]
X=air_pollution_features_df.drop(columns="aqi_6_hour_ahead")

# Slice chronologically using .iloc; 80-20 split
split_idx = int(len(air_pollution_features_df) * 0.8)
X_train,X_test = X.iloc[:split_idx],X.iloc[split_idx:]
Y_train,Y_test = Y.iloc[:split_idx],Y.iloc[split_idx:]

In [35]:
# Count rows where you can guess the aqi 6 hours ahead because it is the same, as a comparison baseline
current_and_six_hour_ahead_same_fraction = ((air_pollution_features_df['aqi'] == air_pollution_features_df['aqi_6_hour_ahead']).sum())/len(air_pollution_features_df)
print(current_and_six_hour_ahead_same_fraction)

0.6061108413678297


In [36]:
model=LGBMClassifier()
model.fit(X_train,Y_train,
         eval_set=[(X_train, Y_train), (X_test, Y_test)],
    eval_names=['train', 'eval'],
    eval_metric='multi_logloss')
Y_pred=model.predict(X_test)
acs=accuracy_score(Y_test,Y_pred)
print(acs*100)

C:\Users\sashe\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.029822 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 10811
[LightGBM] [Info] Number of data points in the train set: 115336, number of used features: 70
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Start training from score -1.364104
[LightGBM] [Info] Start training from score -1.066297
[LightGBM] [Info] Start training from score -1.460642
[LightGBM] [Info] Start training from score -2.468636
[LightGBM] [Info] Start training from score -2.485045
78.31379621280433


In [37]:
train_error_history = model.evals_result_['eval']['multi_logloss']
final_train_error = train_error_history[-1]

print(f"Final Training Error: {final_train_error}")
# train_error_history

Final Training Error: 0.512271793931892


## Model without Point-in-Time Correctness i.e. centred features and leakage

In [38]:
air_pollution_features_df_centred = client.query_df("SELECT * FROM weather_features.feature_air_pollution_pit_correctness_test order by time_of_measurement_utc")
air_pollution_features_df_centred = air_pollution_features_df_centred.dropna()

# Appy one-hot encoding per city and season in the same order every time
air_pollution_features_df_centred['season'] = pd.Categorical(air_pollution_features_df_centred['season'], categories=['Autumn', 'Dry Season', 'Rainy Season', 'Spring', 'Summer', 'Winter'], ordered=True)
air_pollution_features_df_centred['lat_and_long_encoding_id'] = pd.Categorical(air_pollution_features_df_centred['lat_and_long_encoding_id'], categories=latitude_and_longitude_encoding_ids, ordered=True)
air_pollution_features_df_centred = pd.get_dummies(air_pollution_features_df_centred, columns=['lat_and_long_encoding_id'], dtype=int)
air_pollution_features_df_centred = pd.get_dummies(air_pollution_features_df_centred, columns=['season'], dtype=int)

# remove air_pollution_features_row_id. remove time of measurement. 
air_pollution_features_df_centred=air_pollution_features_df_centred.drop(columns="air_pollution_features_row_id")
air_pollution_features_df_centred=air_pollution_features_df_centred.drop(columns="time_of_measurement_utc")
air_pollution_features_df_centred=air_pollution_features_df_centred.drop(columns="time_of_measurement") # relevant time feature

Y2=air_pollution_features_df_centred["aqi_6_hour_ahead"]
X2=air_pollution_features_df_centred.drop(columns="aqi_6_hour_ahead")

# Slice chronologically using .iloc; 80-20 split
split_idx = int(len(air_pollution_features_df_centred) * 0.8)
X_train2,X_test2 = X2.iloc[:split_idx],X2.iloc[split_idx:]
Y_train2,Y_test2 = Y2.iloc[:split_idx],Y2.iloc[split_idx:]

In [39]:
model2=LGBMClassifier()
model2.fit(X_train2,Y_train2,
         eval_set=[(X_train2, Y_train2), (X_test2, Y_test2)],
    eval_names=['train', 'eval'],
    eval_metric='multi_logloss')

C:\Users\sashe\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.028358 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 10806
[LightGBM] [Info] Number of data points in the train set: 115336, number of used features: 70
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Start training from score -1.364104
[LightGBM] [Info] Start training from score -1.066297
[LightGBM] [Info] Start training from score -1.460642
[LightGBM] [Info] Start training from score -2.468636
[LightGBM] [Info] Start training from score -2.485045
82.0038843032531


In [19]:
Y_pred2=model.predict(X_test2)
acs2=accuracy_score(Y_test2,Y_pred2)
print(acs2*100)

train_error_history2 = model2.evals_result_['eval']['multi_logloss']
final_train_error2 = train_error_history2[-1]

Final Training Error: 0.2550705786328779


## Comparison

In [40]:
print("Model with Point-In-Time correctness vs not:")
print(f"Accuracies: {acs} vs {acs2}")
print(f"Final Training Error: {final_train_error} vs {final_train_error2}")

Model with Point-In-Time correctness vs not:
Accuracies: 0.7831379621280433 vs 0.820038843032531


NameError: name 'final_train_error2' is not defined